# 05 — Gold star schema  ·  🟡 STARTER STUB (SA to finish)

The pattern is in place; this is where you (the SA) model the customer-facing star schema. Suggested shape:

- **dim_team** — from `silver_teams`, keyed `team_key`, carrying `external_id_mlbam` (the MLBAM id) so it
  conforms to any other MLBAM-keyed source the customer has.
- **dim_player**, **dim_date** — from `silver_players` / `silver_players_teamhistory`.
- **fact_game** — from `silver_games`, FKs to home/away `dim_team` + `dim_date`.
- **fact_event / fact_pitch** — from `silver_events` / `silver_events_pitch_subset` (the big payloads).

One worked dimension is scaffolded below as a template; fill in the rest, then declare the keys as
**RELY** constraints so AI/BI + Genie can infer the joins (worked example at the bottom).

In [ ]:
# --- dual-mode setup: runs locally via Databricks Connect AND inside a Databricks Git Folder ---
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
try:
    spark  # already present inside a Databricks workspace notebook
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "main")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "synergy_baseball_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
B, S, G = f"{UC_CATALOG}.{BRONZE_SCHEMA}", f"{UC_CATALOG}.{SILVER_SCHEMA}", f"{UC_CATALOG}.{GOLD_SCHEMA}"
print("target:", B, "|", S, "|", G)

In [ ]:
# Worked template: dim_team (carries external_id_mlbam = the MLBAM cross-source join key)
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.dim_team AS
    SELECT
        md5(id)              AS team_key,
        id                   AS synergy_team_id,
        external_id_mlbam    AS mlbam_team_id,   -- <- MLBAM id: the cross-source conformance key
        name, name_abbrev, league_name, division_name, conference_name
    FROM {S}.silver_teams
""")
print("dim_team:", spark.table(f"{G}.dim_team").count())

# RELY PK/FK pattern — declare keys so AI/BI dashboards + Genie can infer joins automatically.
# (NOT NULL is required before a column can be a PRIMARY KEY.)
spark.sql(f"ALTER TABLE {G}.dim_team ALTER COLUMN team_key SET NOT NULL")
spark.sql(f"ALTER TABLE {G}.dim_team ADD CONSTRAINT pk_dim_team PRIMARY KEY (team_key) RELY")
# Then on a fact, e.g.:
#   ALTER TABLE {G}.fact_game ADD CONSTRAINT fk_home_team
#       FOREIGN KEY (home_team_key) REFERENCES {G}.dim_team(team_key) RELY

# TODO(SA): dim_player, dim_date, fact_game (FK home/away team + date), fact_event/fact_pitch.